Thai Legal CCL - RAG Querying Pipeline
Hybrid Search (Dense + Sparse + RRF Fusion + Reranker) with LangChain

### Import

In [1]:
# --- เลือก LLM ที่ต้องการใช้ (uncomment อันที่ต้องการ) ---
# from langchain_openai import ChatOpenAI
# from langchain_anthropic import ChatAnthropic
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
import os
from src.rag.hybridrag_langhchain import ThaiLegalRAG

print("✅ Imports OK")

d:\Github_ThisPC\muict_thai_legal_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports OK


### Configuration

In [10]:
from src.rag.config import RAGConfig


llm = ChatOpenAI(
        model_name="typhoon-v2.5-30b-a3b-instruct",  # หรือรุ่นที่ท่านต้องการใช้
        # model_name="openai/gpt-4o-mini",  # หรือรุ่นที่ท่านต้องการใช้
        openai_api_key = os.getenv("OPENAI_API_KEY"),
        openai_api_base="https://api.opentyphoon.ai/v1",  # สำคัญ: ใส่แทน base_url เดิม
        # openai_api_base="https://openrouter.ai/api/v1",  # สำคัญ: ใส่แทน base_url เดิม
        temperature=0,
        max_tokens=28000,
    )


# --- Build RAG ---
config = RAGConfig(
    retrieval_limit=3,
    reranking_limit=3,
)
rag = ThaiLegalRAG(llm=llm, config=config)
print("✅ RAG chain ready")

[RAG] Running on: CUDA


Fetching 23 files: 100%|██████████| 23/23 [00:00<?, ?it/s]


✅ RAG chain ready


In [10]:
question = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"
answer, _ = rag.chat(question)
print(answer)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาต ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับเพิ่มอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน ตามมาตรา 132 แห่งพระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546


In [4]:
question = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"
output = rag.hybrid_ref_rag(question)
output

{'query': 'ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร',
 'num_candidates': 3,
 'final': 4,
 'docs_candidates': [Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '132', 'reference_laws': [{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '54'}], 'rank': 1, 'score': 7.8828125}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน'),
  Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '54', 'reference_laws': [], 'rank': 2, 'score': 7.8828125}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 54 ศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าต้องเป็นนิติบุคคลประเภทบริษ

In [5]:
question = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"
output = rag.dense_rag(query=question)
output

{'query': 'ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร',
 'num_candidates': 3,
 'final': 3,
 'docs_candidates': [Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '132', 'reference_laws': [{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '54'}], 'rank': 1, 'score': 3}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน'),
  Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '125', 'reference_laws': [{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '16'}], 'rank': 2, 'score': 2}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาต

In [6]:
question = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"
output = rag.sparse_rag(query=question)
output

{'query': 'ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร',
 'num_candidates': 3,
 'final': 3,
 'docs_candidates': [Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '132', 'reference_laws': [{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '54'}], 'rank': 1, 'score': 3}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน'),
  Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '125', 'reference_laws': [{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '16'}], 'rank': 2, 'score': 2}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาต

In [7]:
question = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"
output = rag.hybrid_rag(query=question)
output

{'query': 'ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร',
 'num_candidates': 3,
 'final': 3,
 'docs_candidates': [Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '132', 'reference_laws': [{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '54'}], 'rank': 1, 'score': 3}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน'),
  Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '125', 'reference_laws': [{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '16'}], 'rank': 2, 'score': 2}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาต

In [8]:
question = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"
output = rag.hybrid_rerank_rag(query=question)
output

{'query': 'ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร',
 'num_candidates': 3,
 'final': 3,
 'docs_candidates': [Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '132', 'reference_laws': [{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '54'}], 'rank': 1, 'score': 7.8828125}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน'),
  Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '125', 'reference_laws': [{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '16'}], 'rank': 2, 'score': 6.4453125}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน

In [ ]:
question = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"
docs = rag.retriever.retrieve(query=question, expansion_mode="top-1")
docs

[Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '132', 'reference_laws': [{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '54'}], 'rank': 1, 'score': 7.8828125}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน'),
 Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '54', 'reference_laws': [], 'rank': 2, 'score': 7.8828125}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 54 ศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าต้องเป็นนิติบุคคลประเภทบริษัทมหาชนจำกัดและได้รับใบอนุญาตจากคณะกรรมการ ก.ล.ต.\nศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าที่เปิดให้เฉพาะผู้ลงทุนสถาบันซื้อขายสัญญาซื้อขายล่วงหน้าเพื่อตนเองต้องเป็นนิติบุคคลประเภท

In [7]:
question = "ภาษีเงินได้ที่บริษัทจะต้องชำระ มีอัตราเท่าใด"
docs = rag.retriever.retrieve(query=question, expansion_mode="top-n")
docs

[Document(metadata={'law_name': 'พระราชบัญญัติภาษีเงินได้ปิโตรเลียม พ.ศ. 2514', 'section_num': '65 จัตวา', 'reference_laws': [], 'rank': 1, 'score': 1.654296875}, page_content='พระราชบัญญัติภาษีเงินได้ปิโตรเลียม พ.ศ. 2514 มาตรา 65 จัตวา บริษัทมีหน้าที่เสียภาษีเงินได้ดังต่อไปนี้\n(1)ภาษีเงินได้เป็นรายรอบระยะเวลาบัญชีในอัตราที่กำหนดโดยพระราชกฤษฎีกาซึ่งต้องไม่น้อยกว่าร้อยละ 35 แต่ไม่เกินร้อยละ 48 ของกำไรสุทธิที่ได้จากกิจการปิโตรเลียม\n(2) ภาษีเงินได้ในอัตราร้อยละ23.08 ของเงินกำไรที่เหลือจากการชำระภาษีเงินได้ตาม (1)หรือเงินประเภทอื่นใดที่กันไว้จากกำไรดังกล่าว หรือที่ถือว่าเป็นเงินกำไรดังกล่าว ทั้งนี้ เฉพาะจำนวนที่จำหน่ายออกนอกราชอาณาจักร\nเพื่อประโยชน์ในการคำนวณภาษีเงินได้ตาม(2) ให้ถือภาษีเงินได้ตาม (2) นั้น เป็นเงินกำไรที่จำหน่ายออกนอกราชอาณาจักรด้วย'),
 Document(metadata={'law_name': 'พระราชบัญญัติภาษีเงินได้ปิโตรเลียม พ.ศ. 2514', 'section_num': '45', 'reference_laws': [{'law_name': 'พระราชบัญญัติภาษีเงินได้ปิโตรเลียม พ.ศ. 2514', 'section_num': '21'}], 'rank': 2, 'score': -1.841796875}, 

In [14]:
question = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"
output = rag.hybrid_ref_rag(question, expansion_mode="top-1", reorder_mode="append-last")
output

{'query': 'ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร',
 'num_candidates': 3,
 'final': 4,
 'docs_candidates': [Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '132', 'reference_laws': [{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '54'}], 'rank': 1, 'score': 7.8828125}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน'),
  Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '125', 'reference_laws': [{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '16'}], 'rank': 2, 'score': 6.4453125}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน

In [11]:
question = "ภาษีเงินได้ที่บริษัทจะต้องชำระ มีอัตราเท่าใด"
output = rag.hybrid_ref_rag(question, expansion_mode="top-n", reorder_mode="parent-first")
output

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'query': 'ภาษีเงินได้ที่บริษัทจะต้องชำระ มีอัตราเท่าใด',
 'num_candidates': 3,
 'final': 7,
 'docs_candidates': [Document(metadata={'law_name': 'พระราชบัญญัติภาษีเงินได้ปิโตรเลียม พ.ศ. 2514', 'section_num': '65 จัตวา', 'reference_laws': [], 'rank': 1, 'score': 1.654296875}, page_content='พระราชบัญญัติภาษีเงินได้ปิโตรเลียม พ.ศ. 2514 มาตรา 65 จัตวา บริษัทมีหน้าที่เสียภาษีเงินได้ดังต่อไปนี้\n(1)ภาษีเงินได้เป็นรายรอบระยะเวลาบัญชีในอัตราที่กำหนดโดยพระราชกฤษฎีกาซึ่งต้องไม่น้อยกว่าร้อยละ 35 แต่ไม่เกินร้อยละ 48 ของกำไรสุทธิที่ได้จากกิจการปิโตรเลียม\n(2) ภาษีเงินได้ในอัตราร้อยละ23.08 ของเงินกำไรที่เหลือจากการชำระภาษีเงินได้ตาม (1)หรือเงินประเภทอื่นใดที่กันไว้จากกำไรดังกล่าว หรือที่ถือว่าเป็นเงินกำไรดังกล่าว ทั้งนี้ เฉพาะจำนวนที่จำหน่ายออกนอกราชอาณาจักร\nเพื่อประโยชน์ในการคำนวณภาษีเงินได้ตาม(2) ให้ถือภาษีเงินได้ตาม (2) นั้น เป็นเงินกำไรที่จำหน่ายออกนอกราชอาณาจักรด้วย'),
  Document(metadata={'law_name': 'ประมวลรัษฎากร', 'section_num': '39', 'reference_laws': [{'law_name': 'ประมวลรัษฎากร', 'secti

In [16]:
question = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"
output = rag.hybrid_ref_rag(question, expansion_mode="top-n", reorder_mode="append-last")
output

{'query': 'ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร',
 'num_candidates': 3,
 'final': 6,
 'docs_candidates': [Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '132', 'reference_laws': [{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '54'}], 'rank': 1, 'score': 7.8828125}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน'),
  Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '125', 'reference_laws': [{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '16'}], 'rank': 2, 'score': 6.4453125}, page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน